# $F_{ST}$: how differentiated are two populations?

**Purpose.** $F_{ST}$ measures how much of the genetic variation sits **between**
populations rather than within them. It is the standard summary of population
differentiation, and in conservation genetics it is one of the numbers used to argue that
two groups are separate management units. This exercise computes it two ways, because the
right method depends on whether you can trust your genotypes.

**What you will do**
 - estimate pairwise $F_{ST}$ from **called genotypes** with `plink2`
 - turn the pairwise table into a matrix and read it as a heatmap and a clustering tree
 - relate the values back to where the animals were sampled
 - then estimate $F_{ST}$ from **genotype likelihoods** with site allele frequency
   likelihoods (SAF), for data too low-depth to call

**The data.** Two datasets, both wildlife.

**Blue wildebeest** (*Connochaetes taurinus*) as called genotypes, 95 individuals:

| Group | n |
|---|---|
| Monduli (Tanzania) | 29 |
| Serengeti (Tanzania) | 10 |
| Nairobi (Kenya) | 10 |
| Amboseli (Kenya) | 9 |
| Selous (Tanzania) | 6 |
| Ethosha (Namibia) | 6 |
| Luangwa (Zambia) | 3 |
| **Black wildebeest** (*C. gnou*) | 20 |
| Hartebeest | 2 |

Black wildebeest is a **different species**, included as an outgroup — it should show far
higher $F_{ST}$ than any pair of blue wildebeest, which is the yardstick for reading the
rest. The two hartebeest are removed during the exercise.

**Reindeer** (*Rangifer tarandus*) from Greenland as genotype likelihoods: three
populations — **Qassit**, **Neria** and **Ameralik** — stored as SAF files, with the
pairwise $F_{ST}$ precomputed because the estimation is slow.

Note the sample sizes: 29 from one locality and 3 from another. $F_{ST}$ estimates from
very few individuals are noisy, which is worth remembering when reading the matrix.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/fst

# where you will do the exercise
WORK_DIR=$HOME/fst_animal

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.fst_animal_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/wildebeest_fst.* .
cp -sf $DATA/clusterfile .
cp -r -sf $DATA/reindeer_saf .

echo --programs that are installed:--
which plink2
which realSFS

echo; echo --- files in folder ---
ls

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.fst_animal_workdir"))[1]
setwd(work_d)
getwd()

# $F_{st}$ in wildebeest

We are back to wildebeest!

In this exercise we will cover:
 - Generating and displaying pairwise $F_{st}$ values
    
    
Tools used: plink2, R

The notebooks are editable, so feel free to experiment and change the code to see what happens or write notes in the text cells. Just remember to download the notebooks used here at some point if you want to save them with your own changes included.

We will be using the data set of called genotypes from different blue wildebeest populations, as well as some black wildebeest as an outgroup to compare to, saved in a plink format file set. 

Here is the map from earlier to help show the sampling locations of the different wildebeest populations:
<img src="https://raw.githubusercontent.com/popgenDK/popgenDK.github.io/gh-pages/images/slider/wildeBeastMap.png" alt="image info" />


 **- Do you remember what a plink file set (.bed, bim and .fam) contains?**

In [ ]:
head wildebeest_fst.fam

**Question**
 - The `.fam` file has one line per individual. How many individuals are there in total?

In [ ]:
zstdcat wildebeest_fst.bim.zst | head

**Question**
 - The `.bim` file lists the variants. Why is it compressed with `zstd` here, when the `.fam` is not?

Below we have the command used to run the $F_{st}$ estimation:

In [ ]:
plink2 --bfile wildebeest_fst vzs --within clusterfile \
    --fst CATPHENO method=hudson --allow-extra-chr --threads 10

**Questions**
 - `--within clusterfile` tells plink2 which population each individual belongs to. What would happen without it?
 - Why must $F_{ST}$ be computed *pairwise* rather than as a single number for the whole dataset?

 **- How many individuals are in this files? And divided in how many populations?**

The cluster/within file supplied with the packaged exercise data tells the program how to separate the individuals into different groups for comparison. If we did not know up front which samples belonged together in populations, can you recall something we have looked at that could perhaps help with this?

Then let's have a look at the results:

In [ ]:
# some hartebeest samples were also originally included in this data set, but now we can just remove those from
# the output
grep -ve Hartebeest plink2.fst.summary > tmp
mv tmp plink2.fst.summary

# print the results
column -t plink2.fst.summary

**Questions**
 - How many individuals are in the file, and in how many groups?
 - Two hartebeest were just removed. Why would leaving a second outgroup species in distort the comparison?

 **- Which populations are most genetically differentiated? Which are most similar?**
 
 **- Can you indentify a pattern in the Fst values between black wildebeest and each of the blue wildebeest populations? Try to see if you can explain this pattern.**

Are each of these values large or small? This is quite difficult to answer without context, as it will depend on the type of data you are analyzing, the amount of data and the scope of your study. To provide context, one often looks at a matrix of $F_{st}$ values, which can be visualized using a heatmap. To do this we first need to transform the above data frame into a matrix, and then generate a heatmap using the heatmap.2-function.

In [ ]:
options(repr.matrix.max.cols=10, repr.matrix.max.rows=10)
options(repr.plot.width=16, repr.plot.height=16)
library(gplots)

# read the data into R
fst <- read.table("./plink2.fst.summary")
names(fst) <- c("pop1", "pop2", "est")
fst <- fst[fst$pop1 != "Hartebeest" & fst$pop2 != "Hartebeest",]

**Question**
 - The pairwise table is turned into a matrix here. Which pair has the highest Fst, and does the black wildebeest outgroup behave as you expected?

Here we transform the table from above into a pairwise matrix that contains the exact same information, just in a different format:

In [ ]:
mat <- matrix(NA, 8, 8)
mat[lower.tri(mat)] <- fst$est
mat <- t(mat)
mat[lower.tri(mat)] <- fst$est
colnames(mat) <- c( "Amboseli", fst[1:7,2])
rownames(mat) <- c( "Amboseli", fst[1:7,2])
mat

**Question**
 - The pairwise table is being turned into an 8x8 matrix. Why is only the lower triangle filled in?

In [ ]:
heatmap.2(mat, symm=T, trace='n', cexRow=1.5, cexCol=1.5, margins = c(12, 12))

**- Look at the clustering tree produced by this method. Do the different groups relate to each other as we would expect?**

**- We can see some discrete levels of values in the color key and in the histogram in the inset plot. What do these correspond to?**
 
An important note here is that the tree/dendrogram used to order the groups here simply comes from clustering based on the $F_{st}$ values and will not neccesarily reflect the true evolutionary history of the groups.

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/relatedness_diversity/quiz/quiz_wildebeest_fst.json')

# Extra/optional - Reindeer SAF-based $F_{ST}$ - Using genotype Likelihood

We now estimate genotype-likelihood-based differentiation for **Qassit**, **Neria**, and **Ameralik**. The ANGSD SAF files are supplied in `reindeer_saf`; do not regenerate them.

Each population needs a matching `.saf.idx`, `.saf.gz`, and `.saf.pos.gz` triplet. These SAFs must be full-dimensional (not generated with `ANGSD -fold 1`) and must use the same reference and compatible filters.

`winsfs` estimates the pairwise 2D-SFS. We then remove its `#SHAPE` header and use ANGSD's `realSFS fst` functions for global and windowed $F_{ST}$.

In [ ]:
# the working folder was created and entered in the setup cell
SAF_DIR=reindeer_saf
pops=(Qassit Neria Ameralik)
extensions=(saf.idx saf.gz saf.pos.gz)
missing=0

for pop in "${pops[@]}"; do
  for extension in "${extensions[@]}"; do
    file="$SAF_DIR/$pop.$extension"
    if [[ ! -s "$file" ]]; then
      echo "MISSING: $file" >&2
      missing=1
    fi
  done
done

if (( missing )); then
  echo "Add all nine supplied SAF components to $SAF_DIR before continuing." >&2
  exit 1
fi

echo "All supplied SAF components were found."
ls -lh "$SAF_DIR"/{Qassit,Neria,Ameralik}.saf.{idx,gz,pos.gz}

**Question**
 - These are SAF files rather than genotypes. What does a site allele frequency likelihood store that a called genotype throws away?

In [ ]:
echo "This command was used to generate the precomputed files:"
echo "SAF_DIR=reindeer_saf
OUT=.
THREADS=40

pairs=("Qassit Neria" "Qassit Ameralik" "Neria Ameralik")

for pair in "${pairs[@]}"; do
  read -r pop1 pop2 <<< "$pair"
  prefix="$OUT/${pop1}_${pop2}"
  echo "Estimating $pop1 versus $pop2"

  winsfs --threads "$THREADS" --seed 2026 \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    > "$prefix.winsfs"

  # winsfs writes a #SHAPE header; realSFS fst expects only numeric SFS values.
  tail -n 1 "$prefix.winsfs" > "$prefix.2dsfs"

  realSFS fst index \
    "$SAF_DIR/$pop1.saf.idx" "$SAF_DIR/$pop2.saf.idx" \
    -sfs "$prefix.2dsfs" -fstout "$prefix"

  realSFS fst stats "$prefix.fst.idx" > "$prefix.global_fst.txt"
  realSFS fst stats2 "$prefix.fst.idx" -win 100000 -step 50000 \
    > "$prefix.100kb_fst.tsv"
done"

echo
echo "Estimating Qassit versus Neria
Estimating Qassit versus Ameralik
Estimating Neria versus Ameralik"

In [ ]:
echo ----Global pairwise Fst results----
for result in reindeer_saf/*.global_fst.txt; do
  echo "Population Pair  100kb    global"
  printf '%s: ' "$(basename "$result" .global_fst.txt)"
  cat "$result"
  echo
done

**Questions**
 - Which reindeer pair has the largest global $F_{ST}$, and which the smallest?
 - The wildebeest values were computed from called genotypes and these from likelihoods. Would you expect the two approaches to give the same number on the same data?

**Which population pair has the largest global $F_{ST}$? Which has the smallest?**

**Do particular 100-kb windows show much stronger differentiation than the global estimate?**

**Why must the three SAF datasets use compatible genomic sites, references, and filters?**

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/relatedness_diversity/quiz/quiz_reindeer_saf_fst.json')